# Elastic Network Adapter (ENA)

A practical refresher on **Elastic Network Adapter (ENA)** — the AWS *enhanced networking* interface that gives EC2 instances high bandwidth, low latency, and low CPU overhead. ENA is the default high-performance NIC behind almost every modern instance type and the foundation that **EFA** (OS-bypass) extends.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Elastic Network Adapter (ENA) is the **enhanced-networking device** AWS exposes to EC2 instances. It is a custom network interface (driver `ena` on Linux, with an SR-IOV virtual function) that delivers far higher packet-per-second rates and bandwidth, and far lower latency and jitter, than the older paravirtualized or Intel `ixgbevf` networking. On current-generation instances ENA is simply *how* the instance talks to the network — there is no separate "turn on networking" step beyond having the driver and the `EnaSupport` flag enabled.

### What is it?

ENA is a hardware network interface, offloaded to the **AWS Nitro system**, presented to the guest as an SR-IOV virtual function. The guest loads the open-source `ena` Linux kernel driver (or the ENA driver on Windows/DPDK), which talks directly to the Nitro card. Because the data path is hardware-offloaded rather than emulated in the hypervisor, ENA achieves line-rate throughput with low CPU cost. Depending on instance size it provides anywhere from a few Gbps up to **100, 200, or 400 Gbps** of network bandwidth.

ENA also has an enhancement, **ENA Express**, which layers the AWS **SRD (Scalable Reliable Datagram)** transport under ordinary TCP/UDP traffic — automatically multipathing packets to raise single-flow bandwidth and lower tail latency without any application change.

### Why use it?

- **High bandwidth** — up to 100/200/400 Gbps on large instances, versus single-digit Gbps on legacy networking.
- **Low latency & jitter** — hardware offload on Nitro keeps the data path off the hypervisor.
- **Low CPU overhead** — multi-queue + RSS spread interrupts across vCPUs, freeing cores for compute.
- **No extra cost / no extra setup** — ENA is included with supported instances; modern AMIs ship the driver.
- **ENA Express** — opt-in SRD acceleration that improves single-flow throughput and p99 latency for unmodified apps.

### When to use it?

- Essentially **always** on current-generation instances — it is the default and recommended NIC.
- **Data-heavy ML pipelines**: streaming training shards from S3, loading from FSx/EBS, feature stores — all benefit from ENA's throughput.
- **High packet-rate services**: inference servers, proxies, and load balancers handling many small requests.
- Single-node or loosely-coupled multi-node work that needs fast TCP but not OS-bypass (use **EFA** when you need RDMA-class collective performance for tightly-coupled training).

## Key Features

### Core Capabilities of Elastic Network Adapter (ENA)

| Feature | Description | Benefit |
|---------|-------------|---------|
| SR-IOV hardware offload | Network device is a Nitro VF, not hypervisor-emulated | Line-rate throughput at low CPU cost |
| Up to 400 Gbps | Bandwidth scales with instance size (multiple network cards on big types) | Saturate S3/EBS and inter-node TCP |
| Multi-queue + RSS | Multiple TX/RX queues, Receive Side Scaling hashes flows across vCPUs | Parallel packet processing, no single-core bottleneck |
| ENA Express (SRD) | SRD multipath transport under TCP/UDP, opt-in per ENI | Higher single-flow BW, lower p99 latency, no app change |
| Jumbo frames (9001 MTU) | Large MTU within the VPC | Fewer packets/interrupts for bulk transfer |
| Driver-reported metrics | Allowance counters exposed via `ethtool -S` | Detect when instance network limits are throttling you |
| DPDK / poll-mode support | ENA PMD for userspace packet processing | Very high PPS for NFV / packet-heavy workloads |
| Foundation for EFA | EFA is an ENA with an added OS-bypass path | One card serves both TCP and the HPC fabric |

## Architecture Overview

ENA sits between the guest kernel and the AWS Nitro networking hardware. The guest `ena` driver posts packets to hardware queues exposed as an SR-IOV virtual function; the Nitro card handles the actual VPC networking, encapsulation, and (optionally) ENA Express / SRD.

```
   Application  (training data loader, inference server, MPI/NCCL over TCP)
        |  sockets (TCP/UDP)
   Linux network stack
        |
   ena kernel driver  ---- multiple TX/RX queues, RSS across vCPUs
        |  SR-IOV virtual function
   Nitro card  (hardware offload: VPC encap, security groups, metering)
        |  [ ENA Express: SRD multipath under TCP/UDP — optional ]
        v
   VPC network  ->  S3 / EBS / FSx / peer EC2 instances

   (EFA adds an OS-bypass Libfabric path alongside this same ENA interface.)
```

### Components

1. **ENA device (Nitro VF)**: the SR-IOV virtual function presented to the instance; the data path is offloaded to the Nitro card.
2. **`ena` kernel driver**: the open-source Linux driver (also Windows/FreeBSD and a DPDK PMD) that manages the hardware queues and reports counters.
3. **Multi-queue + RSS**: several TX/RX queue pairs with Receive Side Scaling so flows are spread across vCPUs for parallelism.
4. **Allowance metering**: the Nitro card enforces per-instance bandwidth, PPS, and connection-tracking limits, exposing how close you are via driver counters.
5. **ENA Express / SRD**: an optional transport layer that sprays packets across paths to raise single-flow bandwidth and cut tail latency.

## Installation

### Prerequisites

- A **current-generation, ENA-capable instance type** (virtually all are: C5/C6/C7, M5/M6/M7, R5/R6/R7, P3/P4/P5, G4/G5, etc.).
- The instance attribute **`EnaSupport=true`** (default on supported types; required for the device to appear).
- The **`ena` driver** in the AMI. All current Amazon Linux 2/2023, Ubuntu, and AWS Deep Learning AMIs ship it. Custom/old AMIs may need the driver installed and `EnaSupport` enabled before the network device works.

### Installation Steps

On a modern AMI there is nothing to install — verify instead. The cell below checks the driver and version. If you build a custom AMI from an old base, you install the driver from source and set the instance attribute (commands shown commented).

In [ ]:
# Verify ENA is active on this host (works on any Linux EC2 instance with ENA).
# Off EC2 this just reports that no ena device is present.
import glob
import os
import subprocess

def first_iface():
    """Return the primary non-loopback interface name, if any."""
    names = [os.path.basename(p) for p in glob.glob("/sys/class/net/*")]
    return next((n for n in names if n != "lo"), None)

def driver_for(iface):
    link = f"/sys/class/net/{iface}/device/driver"
    return os.path.basename(os.path.realpath(link)) if os.path.exists(link) else None

iface = first_iface()
if iface and driver_for(iface) == "ena":
    print(f"{iface}: ENA driver active.")
    try:
        out = subprocess.run(["ethtool", "-i", iface], capture_output=True, text=True, timeout=5)
        for line in out.stdout.splitlines():
            if line.startswith(("driver", "version", "firmware")):
                print("  " + line)
    except Exception:
        pass
else:
    print(f"No ENA device here (iface={iface}, driver={driver_for(iface) if iface else None}).")
    print("Expected off an ENA-enabled EC2 instance; the checks apply on one.")

# Building a custom AMI from an old base (run on the instance, then set the attribute):
#   git clone https://github.com/amzn/amzn-drivers.git
#   cd amzn-drivers/kernel/linux/ena && make && sudo insmod ena.ko
#   # from an admin host, with the instance stopped:
#   aws ec2 modify-instance-attribute --instance-id i-xxxx --ena-support

## Basic Usage

### Quick Start Example

You don't program ENA directly — you use normal sockets and the kernel uses ENA. What matters operationally is knowing your instance's **network allowance** (bandwidth/PPS limits) and whether you are hitting it. The cell below reads the ENA driver's allowance counters via `ethtool -S`, which is the canonical way to tell whether the *instance's network limit* is throttling your traffic.

In [ ]:
# Read ENA allowance counters. When these "exceeded" counters climb, the Nitro
# card is shaping your traffic because you hit an instance network limit.
import subprocess

ALLOWANCE_COUNTERS = (
    "bw_in_allowance_exceeded",    # inbound bandwidth cap hit
    "bw_out_allowance_exceeded",   # outbound bandwidth cap hit
    "pps_allowance_exceeded",      # packets-per-second cap hit
    "conntrack_allowance_exceeded",# tracked-connections cap hit
    "linklocal_allowance_exceeded",# metadata/DNS/NTP request cap hit
)

def ena_allowance(iface):
    out = subprocess.run(["ethtool", "-S", iface], capture_output=True, text=True, timeout=5)
    stats = {}
    for line in out.stdout.splitlines():
        if ":" in line:
            k, _, v = line.strip().partition(":")
            stats[k.strip()] = v.strip()
    return {c: stats.get(c) for c in ALLOWANCE_COUNTERS if c in stats}

try:
    counters = ena_allowance("eth0")
    if counters:
        print("ENA allowance counters (nonzero = you hit an instance network limit):")
        for k, v in counters.items():
            print(f"  {k:30s} {v}")
    else:
        print("No ENA allowance counters found (not an ENA instance, or different iface name).")
except Exception as exc:
    print(f"Not on an ENA host: {exc}")

### Measuring real throughput

Synthetic point-to-point throughput between two ENA instances is easy to check with `iperf3`. Run it **inside a placement group / same AZ** to see the instance's true allowance, and use **parallel streams** (`-P`) because a single TCP flow may not fill a 100 Gbps link without ENA Express.

```bash
# On the server instance:
iperf3 -s

# On the client instance (8 parallel streams to a 100 Gbps-class target):
iperf3 -c <server-private-ip> -P 8 -t 30
# Sum the streams; compare against the instance type's advertised bandwidth.
# If one stream is far below the aggregate, single-flow is the limit -> consider ENA Express.
```

## Advanced Features

### ENA Express, multiple network cards, and queue tuning

#### ENA Express (SRD under TCP/UDP)

ENA Express transparently runs your TCP/UDP traffic over the **SRD** transport — the same multipath, hardware-congestion-controlled protocol EFA uses — without any application change. It raises **single-flow** bandwidth (a single TCP connection can reach ~25 Gbps) and reduces **tail latency** (p99). You enable it per network interface (console, CLI, or launch template); both endpoints must have it on and be in the same subnet/placement group. It is ideal when your workload can't easily open many parallel flows.

#### Multiple network cards

The largest instances (e.g. `p5.48xlarge`, `c6gn`/`hpc` types) expose **several physical network cards**, each its own ENA, to aggregate bandwidth (e.g. 4 cards × 100 Gbps = 400 Gbps). You attach one ENI per `NetworkCardIndex`; the OS sees multiple interfaces and the workload (or routing) spreads traffic across them.

#### Queue and RSS tuning

ENA exposes multiple TX/RX queues with RSS. Defaults are usually right, but you can inspect and tune them. The cell below shows how to read the queue/ring configuration.

In [ ]:
# Inspect ENA multi-queue + ring configuration (read-only; safe to show anywhere).
import subprocess

def show_queues(iface="eth0"):
    try:
        ch = subprocess.run(["ethtool", "-l", iface], capture_output=True, text=True, timeout=5)
        rg = subprocess.run(["ethtool", "-g", iface], capture_output=True, text=True, timeout=5)
        print(f"--- {iface} channels (queues) ---\n{ch.stdout}")
        print(f"--- {iface} ring sizes ---\n{rg.stdout}")
    except Exception as exc:
        print(f"Not on an ENA host: {exc}")

show_queues()

# Tuning examples (apply on the instance; defaults are good for most workloads):
#   sudo ethtool -L eth0 combined 16     # set 16 combined queues (<= vCPU count)
#   sudo ethtool -G eth0 rx 8192 tx 8192 # enlarge rings to absorb bursts
#   # Ensure irqbalance or RPS spreads RX interrupts across vCPUs.
print("\nQueues should generally match vCPU count so RSS can parallelize RX/TX.")

## Use Cases

### Real-world Applications of Elastic Network Adapter (ENA)

#### Use Case 1: High-throughput training data ingestion

- **Context**: A GPU training job streams sharded datasets (WebDataset/TFRecord/Parquet) from S3 or FSx for Lustre and must keep the GPUs fed.
- **Implementation**: An ENA instance with enough network allowance; many parallel S3 connections (e.g. `s5cmd`, boto3 with a large connection pool) to exploit multi-queue/RSS.
- **Results**: Data loading saturates the link instead of a single core, so GPUs stay busy and step time isn't I/O-bound.

#### Use Case 2: Low-latency, high-PPS inference serving

- **Context**: A model server (Triton, TorchServe, vLLM) fielding many small concurrent requests behind a load balancer.
- **Implementation**: ENA's multi-queue + RSS spreads interrupts across vCPUs; tune queue count to vCPUs; monitor `pps_allowance_exceeded`.
- **Results**: High request throughput with low, stable latency and headroom on the CPU for inference work.

#### Use Case 3: Single-flow-bound transfers accelerated with ENA Express

- **Context**: A pipeline moves large artifacts (checkpoints, datasets) over a small number of TCP connections and can't easily parallelize.
- **Implementation**: Enable **ENA Express** on both ENIs within the same subnet/placement group.
- **Results**: Single-flow bandwidth jumps (~25 Gbps) and p99 latency drops, with no code change.

## Best Practices

### Recommended Practices for Elastic Network Adapter (ENA)

1. **Use current-generation instances and modern AMIs** — they ship the `ena` driver and enable enhanced networking by default; no manual setup needed.
2. **Keep the ENA driver reasonably current** — newer `ena` driver versions add counters, ENA Express support, and performance fixes; the DLAMI tracks them.
3. **Parallelize your traffic** — open multiple TCP flows (or enable ENA Express) so you can fill large links; a single flow won't saturate 100 Gbps on its own.
4. **Right-size the instance to the network need** — bandwidth scales with instance size; if you're network-bound, a bigger instance (or one with multiple network cards) raises the allowance.
5. **Watch the allowance counters** — alarm on `*_allowance_exceeded` so you learn when the *instance limit*, not your code, is the bottleneck.
6. **Co-locate communicating instances** — same AZ and a cluster placement group minimize latency and maximize inter-instance bandwidth.

## Common Pitfalls

### What to Avoid When Using Elastic Network Adapter (ENA)

1. **Blaming the app for a throttle** — flat throughput often means you hit the instance's bandwidth/PPS allowance, not a code bug. Check `bw_*_allowance_exceeded` / `pps_allowance_exceeded` first.
2. **Single-flow expectations** — assuming one TCP stream will reach the advertised aggregate. Use parallel streams or ENA Express.
3. **Old/custom AMI without the driver** — a custom AMI built from an old base may lack the `ena` driver or have `EnaSupport=false`, so the instance falls back to slow networking or fails to get the device. Install the driver and set the attribute.
4. **Confusing ENA with EFA** — ENA accelerates *TCP/IP*; it does **not** give NCCL/MPI the OS-bypass RDMA path. Tightly-coupled multi-node training that's comms-bound needs **EFA**.
5. **Ignoring connection-tracking limits** — services with huge numbers of concurrent connections can hit `conntrack_allowance_exceeded`; security-group rules that mark traffic as "untracked" can help.
6. **Mismatched ENA Express** — it only works when **both** endpoints have it enabled and are in the same subnet/placement group; one-sided config silently gives no benefit.

## Performance Optimization

### Optimizing Elastic Network Adapter (ENA) for Production

#### Configuration Tuning

Key levers, roughly in order of impact:

- **Instance size / network allowance**: bandwidth and PPS scale with the instance; pick a type whose allowance exceeds your peak need (or one with multiple network cards).
- **Parallel flows or ENA Express**: spread load across many connections, or enable ENA Express to lift single-flow bandwidth and cut tail latency.
- **Queue count = vCPUs + RSS**: ensure RX/TX queues match vCPUs so RSS parallelizes packet processing; verify interrupts are balanced across cores.
- **Ring sizes**: enlarge RX/TX rings (`ethtool -G`) to absorb bursts and avoid drops under load.
- **Jumbo frames (MTU 9001)**: within the VPC, fewer/larger packets reduce per-packet overhead for bulk transfer.
- **Placement group + same AZ**: minimizes latency and maximizes inter-instance bandwidth.

The cell below sketches a simple, dependency-free throughput check you can run between two instances to confirm you're near the expected allowance before committing to a configuration.

In [ ]:
# Lightweight throughput self-test over a TCP socket (loopback demo here; point
# host/port at a second instance in production). Real benchmarking should use iperf3.
import socket
import time

PAYLOAD = b"x" * (1 << 20)  # 1 MiB
CHUNKS = 64                  # ~64 MiB total

def serve_once(host="127.0.0.1", port=0):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((host, port)); srv.listen(1)
    return srv, srv.getsockname()[1]

def measure(host, port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect((host, port))
    t0 = time.perf_counter()
    for _ in range(CHUNKS):
        s.sendall(PAYLOAD)
    s.shutdown(socket.SHUT_WR); s.recv(1); s.close()
    return CHUNKS * len(PAYLOAD) / (time.perf_counter() - t0)

import threading
srv, port = serve_once()
def _drain():
    conn, _ = srv.accept()
    while conn.recv(1 << 16):
        pass
    conn.sendall(b"1"); conn.close()
threading.Thread(target=_drain, daemon=True).start()

bps = measure("127.0.0.1", port); srv.close()
print(f"Loopback throughput: {bps/1e9:.1f} GB/s (demo only).")
print("On two ENA instances, prefer: iperf3 -c <ip> -P 8 -t 30 and compare to the allowance.")

## Production Deployment

### Deploying Elastic Network Adapter (ENA) in Production

You don't deploy ENA itself — you ensure instances have enhanced networking and (optionally) ENA Express, the right placement, and enough allowance. Below are the common paths.

#### Launch template with enhanced networking + ENA Express (Terraform)

```hcl
resource "aws_placement_group" "cluster" {
  name     = "ml-net"
  strategy = "cluster"            # co-locate for low latency / high BW
}

resource "aws_launch_template" "ena" {
  instance_type = "c6in.32xlarge" # 200 Gbps-class, ENA-capable
  image_id      = data.aws_ami.al2023.id  # modern AMI -> ena driver included

  network_interfaces {
    device_index          = 0
    security_groups       = [aws_security_group.app.id]
    delete_on_termination = true
    # Enable ENA Express (SRD under TCP/UDP) on this interface:
    ena_srd_specification {
      ena_srd_enabled = true
      ena_srd_udp_specification { ena_srd_udp_enabled = true }
    }
  }
  placement { group_name = aws_placement_group.cluster.name }
}
```

#### Enabling enhanced networking on an existing/custom AMI (CLI)

```bash
# The instance must be stopped. Enables the ENA device on next start.
aws ec2 modify-instance-attribute --instance-id i-0123456789abcdef0 --ena-support
# Confirm:
aws ec2 describe-instances --instance-ids i-0123456789abcdef0 \
  --query 'Reservations[].Instances[].EnaSupport'
```

#### Kubernetes (EKS) networking note

The **AWS VPC CNI** attaches ENA-backed ENIs to nodes and assigns pod IPs from the VPC, so pods get native ENA performance automatically. For very high pod density, raise the ENI/IP allowance (prefix delegation):

```bash
kubectl set env daemonset aws-node -n kube-system ENABLE_PREFIX_DELEGATION=true
```

> Managed services (SageMaker, ECS/EKS on Nitro instances) configure ENA for you; you mainly choose an instance type with enough network allowance.

## Monitoring and Observability

### Monitoring Elastic Network Adapter (ENA) in Production

#### Key Metrics to Track

The ENA driver surfaces rich counters via `ethtool -S eth0`; the **allowance** counters are the ones that explain throttling:

- **`bw_in_allowance_exceeded` / `bw_out_allowance_exceeded`**: number of times inbound/outbound bandwidth was shaped — the clearest sign you hit the instance bandwidth cap.
- **`pps_allowance_exceeded`**: packet-per-second cap hits — common for high-PPS, small-packet services.
- **`conntrack_allowance_exceeded`**: tracked-connection cap hits — relevant for services with many concurrent connections.
- **`linklocal_allowance_exceeded`**: throttling of requests to link-local services (instance metadata, DNS, NTP).
- **`*_queue_*` drops / `tx_timeout`**: queue overruns or stalls suggesting ring/queue tuning is needed.
- **CloudWatch `NetworkIn`/`NetworkOut`/`NetworkPacketsIn`/`Out`**: instance-level traffic for dashboards and alarms.

#### Logging Best Practices

- Periodically scrape `ethtool -S eth0` and ship the **allowance counters to CloudWatch** as custom metrics; alarm when any `*_allowance_exceeded` increases.
- Correlate network throttling with **application latency / GPU utilization** so you can distinguish "instance too small" from "app inefficiency."
- Record the **`ena` driver and firmware version** (`ethtool -i eth0`) with each deployment for reproducibility.
- For EKS, monitor **ENI/IP exhaustion** (VPC CNI metrics) alongside ENA counters.

## Troubleshooting

### Common Issues with Elastic Network Adapter (ENA)

#### Issue 1: Throughput plateaus well below the link rate

**Symptoms**: Bandwidth flatlines; latency rises under load.

**Cause**: You hit the instance's bandwidth/PPS allowance, or you're using a single TCP flow.

**Solution**: Check `bw_*_allowance_exceeded` / `pps_allowance_exceeded` via `ethtool -S`. Use parallel flows or enable ENA Express; if the allowance itself is the cap, move to a larger instance or one with multiple network cards.

#### Issue 2: No ENA device after launching from a custom AMI

**Symptoms**: Slow networking, or the expected interface/driver is missing (`driver` is not `ena`).

**Cause**: The AMI lacks the `ena` driver, or `EnaSupport` is false on the instance/AMI.

**Solution**: Install the `ena` driver from `amzn-drivers`, then (with the instance stopped) run `aws ec2 modify-instance-attribute --ena-support` and restart. Verify with `ethtool -i eth0`.

#### Issue 3: Many connections fail or stall intermittently

**Symptoms**: New connections drop under high concurrency; `conntrack_allowance_exceeded` climbing.

**Cause**: The instance's connection-tracking allowance is exhausted.

**Solution**: Reduce concurrent tracked connections, scale out, or configure security-group rules so high-volume flows are **untracked** (e.g. broad allow rules), and consider a larger instance with a higher allowance.

## Comparison with Alternatives

### How Elastic Network Adapter (ENA) Compares to Other Solutions

| Dimension | ENA | ENA Express | EFA | Legacy (ixgbevf / PV) |
|-----------|-----|-------------|-----|-----------------------|
| Data path | SR-IOV, kernel TCP/IP | TCP/UDP over SRD | OS-bypass (Libfabric/SRD) | Emulated / older SR-IOV |
| Max bandwidth | up to 400 Gbps | same, better single-flow | up to 3,200 Gbps (multi-NIC) | single-digit Gbps |
| Single-flow BW | limited by one path | ~25 Gbps per flow | full (multipath) | low |
| Latency | low | lower p99 | lowest (RDMA-class) | high, variable |
| App changes | none | none (opt-in toggle) | use NCCL/MPI over Libfabric | none |
| Best for | general high-perf TCP | single-flow-bound TCP/UDP | tightly-coupled multi-node HPC/ML | nothing new (deprecated) |

### When to Choose This Tool

Choose **ENA** when:

- You want **high-throughput, low-latency TCP/IP** networking — which is essentially every modern EC2 workload.
- Your traffic is **data movement or request/response** (S3/EBS/FSx ingestion, inference serving), not latency-bound collective communication.
- You can parallelize flows or turn on **ENA Express** for single-flow-bound transfers.

Prefer **EFA** for comms-bound, tightly-coupled multi-node training/HPC (NCCL/MPI all-reduce), and you'd only see **legacy networking** on old instance types or AMIs — migrate off them.

## Resources

### Official Documentation

- Enhanced networking with ENA (EC2 User Guide): https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/enhanced-networking-ena.html
- ENA Express: https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ena-express.html
- Instance network bandwidth & allowances: https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ec2-instance-network-bandwidth.html
- Monitoring ENA network performance (allowance counters): https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/monitoring-network-performance-ena.html

### Tutorials and Guides

- ENA Linux driver (amzn-drivers, GitHub): https://github.com/amzn/amzn-drivers
- ENA DPDK poll-mode driver: https://doc.dpdk.org/guides/nics/ena.html
- AWS VPC CNI for Kubernetes: https://github.com/aws/amazon-vpc-cni-k8s
- Networking blog — ENA Express deep dive: https://aws.amazon.com/blogs/networking-and-content-delivery/

### Community Resources

- AWS re:Post (networking Q&A): https://repost.aws/tags/questions
- AWS Compute & HPC blogs: https://aws.amazon.com/blogs/hpc/
- Stack Overflow tag: https://stackoverflow.com/questions/tagged/amazon-ec2

### Related Technologies

- EFA (Elastic Fabric Adapter) — ENA plus an OS-bypass RDMA path for HPC/ML collectives
- AWS Nitro System — the hardware that offloads ENA networking
- SRD (Scalable Reliable Datagram) — the multipath transport behind ENA Express and EFA